# 4 · Frequency translation and decimation

**Local Quartus workflow · host or container · one active FPGA design at a time**

`z[n]=x[n]·exp(−j2πf₀n/fₛ),  y[m]=(h*z)[mM]`

**What is added:** Frequency translation and decimation: a digital downconverter.

**Why it is useful:** Move a wanted signal to the center, reject a neighbor and reduce the output rate without aliasing.

**How it works:** An FPGA phase accumulator and complex mixer shift frequency; separate I/Q FIR II filters remove unwanted energy before keeping every Mth sample. A clean FPGA two-tone pattern makes the effect easy to see.

```text
IQ → NCO / mixer → I/Q filters → decimator
```

Python model → native VDS diagram → real RTL/IP simulation → SOF → board → focused CyberEther page.

## Setup

Once, from this example folder: `bash scripts/install_local.sh --system` (Linux system packages need sudo), then select **Atum A3 IQ Demos** as the notebook kernel. Quartus/Questa/Nios tools must already be installed. Set the optional paths below if they are not on PATH. No Docker commands are needed.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
# Optional overrides, set before tool discovery:
# os.environ["QUARTUS_ROOTDIR"] = "/your/install/quartus"
# os.environ["QUESTA_ROOT"] = "/your/install/questa_fse"
# os.environ["ATUM_ALTERA_ROOT"] = "/your/install"
# os.environ["ATUM_UART"] = "/dev/serial/by-id/your-board"
# os.environ["ATUM_JTAG_CABLE"] = "your cable from jtagconfig"
from lessons import Lesson, lan_ip, stop_demo
from common import tool_exe
CHECKPOINT = 4
FORCE_REBUILD = False   # Fresh checkout builds; source-matched SOFs are reused afterwards.
lesson = Lesson(CHECKPOINT)
HOST = lan_ip()
print("Python:", sys.executable)
print("Quartus:", tool_exe("quartus_sh"))
print("Questa:", tool_exe("vsim"))
print("Browser host:", HOST)


## 1 · Python simulation

The code below is the editable numerical model. Its plots are synthetic/theory, separate from the RTL results and live hardware below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from notebook_plots import spectrum
from nios_fir import PROFILES, fixed_reference
from nios_ddc import reference, gain_reference
from iq_lab_samples import load_sample
raw, _ = load_sample("two-tone")
fs = 16e6
n = CHECKPOINT
filtered = reference(raw, 134217728, 4, 0)['result'].astype(float) / 128
alias = reference(raw, 134217728, 4, 2)['result'].astype(float) / 128
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
spectrum(axs[0], raw.astype(float), fs, 'Before translation')
spectrum(axs[1], alias[32:], fs / 4, 'Shift + decimate, no filter')
spectrum(axs[2], filtered[32:], fs / 4, 'Shift + FIR + decimate')
axs[1].set_title('Interferer folds into output band')
axs[2].set_title('FIR rejects before downsampling')
summary = {'NCO_phase_increment': '0x08000000', 'shift_hz': 500000, 'decimation': 4, 'nominal_output_rate': 4000000, 'startup_output_samples_omitted': 32}
fig.suptitle(f"{CHECKPOINT} · Python simulation / integer reference")
fig.tight_layout()
plt.show()
summary


## 2 · Generate the actual FPGA block design

Rebuild the Nios initialization memory where present, recreate the native VDS system, validate and generate vendor IP. The cell opens **Quartus Visual Designer Studio itself**, fits the diagram and uses **File → Export**. Its native PDF is rendered to a high-resolution PNG and displayed inline. The original PDF and a full GUI screenshot are saved beside it. A private X display keeps capture separate from your desktop. The output also provides **Quartus project** and **complete project ZIP** links. The ZIP contains the SDC, RTL, generated IP and Nios memory initialization. Extract it before opening the QPF in Quartus.

Timing constraints: [iq_bridge_vds.sdc](../fpga_hub/iq_bridge/vds_demo_04_ddc/iq_bridge_vds.sdc).

In [ ]:
from IPython.display import Image, display
diagram_png = lesson.generate()
display(Image(filename=str(diagram_png)))
print("Exported:", diagram_png)

### Saved native VDS GUI diagram

![Checkpoint 4 — actual VDS GUI export](diagrams/04_vds_gui.png)

The cell above refreshes this diagram from your local design.

### Official Nios V BSP and applications

Generate a HAL BSP from this actual VDS system, plus editable **Hello World** and **IQ monitor** CMake applications. This builds software only; it does not download or replace the running firmware. Open both BSP and app with **RiscFree → File → Import Nios V CMake Project**, then build with Ctrl+B. See [Nios V IDE guide](../software/README.md) for BSP Editor, JTAG download and restore steps.

In [ ]:
software_dir = lesson.software()
print("BSP:", software_dir / "bsp")
print("Hello World:", software_dir / "hello_world")
print("IQ monitor:", software_dir / "iq_monitor")

## 3 · RTL/IP simulation

Actual generated I/Q FIR II plus DDC/snapshot RTL, four mixer/filter/decimation cases.

The plots below read the actual simulator output files. Assertions must pass before the result is plotted.

In [ ]:
simulation_result = lesson.simulate()
print("Simulation:", simulation_result["status"])
lesson.simulation_plot()

### Open the Questa GUI

Use a desktop terminal **on the machine running Quartus** (or its container desktop). Execute the printed commands. `wave.wlf` opens the saved real simulation; `view.do` adds its signals. To rerun interactively, start `vsim -gui` in the same directory and type `do gui.do` in the Transcript. This reruns the same assertions and leaves the waveform window open.

In [ ]:
lesson.questa_gui()

## 4 · Build the SOF

A source-matched, hash-verified cached bitfile can be reused for a quick presentation. Set `FORCE_REBUILD=True` to demonstrate a full Quartus compilation. A fresh checkout always requires a build. Both a new build and cached reuse display links to the complete Quartus project and its ZIP. Open the linked QPF (or extract the ZIP first), then use **Processing → Start Compilation** to build in the GUI.

In [ ]:
build = lesson.build(force=FORCE_REBUILD)
print("SOF:", lesson.stage / "iq_bridge_vds.sof")
print("SHA256:", build["sha256"])
print("Compiled resources:", build["resources"])

## 5 · Program the physical board

This cell **programs the selected checkpoint**. It first stops a previous notebook-owned demo and releases UART/JTAG. Stop any unrelated acquisition service before running it; an unknown serial owner is refused. No ESP32 flashing or reset-button presses are needed with the installed suite.

In [ ]:
lesson.program()

## 6 · Focused CyberEther demo

Select Shift + filter + decimate, then no filter. The interferer folds into the output band when anti-alias filtering is bypassed.

The page uses the bitfile just programmed. One board and one acquisition owner are shared across all notebooks.

In [ ]:
live = lesson.start_demo(host=HOST)
print(live)

Leave the page running while presenting. The next notebook stops it before programming. To stop it manually, run `stop_demo()` in the cell below. Live invites and captures stay in ignored runtime folders; do not publish their tokens.

In [ ]:
# stop_demo()